In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 16:04:04.347633: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-07 16:04:04.384376: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 16:04:05.324101: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 16:04:07,124 [DEBUG] [Rain] Rain is initialized
2023-07-07 16:04:07,132 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 16:04:07,137 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 16:04:07,141 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 16:04:07,147 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 16:04:07,156 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:04:07,165 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 16:04:07,173 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-07 16:04:07,196 [INFO] [Provisioner] provisioner is serving
2023-07-07 16:04:07,197 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 16:04:07,199 [INFO] [Coordinator] coordinator is serving
2023-07-07 16:04:07,201 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 16:04:07,206 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:04:07,208 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 16:04:07,212 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-07 16:04:07,220 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-07 16:04:07,223 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 16:04:07,233 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 16:04:07,240 [INFO] [Worker_50151] Worker is running 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 5s 10ms/step - loss: 0.6870 - accuracy: 0.7821
Epoch 2/5
157/157 [==============================] - 2s 10ms/step - loss: 0.3066 - accuracy: 0.9084
Epoch 3/5
157/157 [==============================] - 2s 11ms/step - loss: 0.3091 - accuracy: 0.9085
Epoch 3/5
157/157 [==============================] - 2s 10ms/step - loss: 0.2378 - accuracy: 0.9299
Epoch 4/5
157/157 [==============================] - 2s 10ms/step - loss: 0.2285 - accuracy: 0.9329
Epoch 4/5
157/157 [==============================] - 2s 11ms/step - loss: 0.1925 - accuracy: 0.9411
Epoch 5/5
157/157 [==============================] - 2s 11ms/step - loss: 0.1951 - accuracy: 0.9431
Epoch 5/5
157/157 [==============================] - 2s 11ms/step - loss: 0.1919 - accuracy: 0.9434
Epoch 5/5
157/157 [==============================] - 2s 10ms/step - loss: 0.1646 - accuracy: 0.9520


2023-07-07 16:04:21,969 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3


sending data to divider


2023-07-07 16:04:21,971 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-07 16:04:22,050 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 16:04:22,069 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
2023-07-07 16:04:22,102 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 3.
2023-07-07 16:04:22,104 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-07 16:04:22,105 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-07 16:04:22,106 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 3
2023-07-07 16:04:22,107 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/3.pkl to worker3
2023-07-07 16:04:22,184 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-07 16:04:22,186 [DEBUG] [DividerAmbassador] divider begins execu

...

2023-07-07 16:04:47,254 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2


157/157 [==============================] - 1s 8ms/step - loss: 0.0930 - accuracy: 0.9707


2023-07-07 16:04:47,373 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.
2023-07-07 16:04:47,386 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 16:04:47,390 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-07 16:04:47,490 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 16:04:47,518 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-07 16:04:47,564 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.
2023-07-07 16:04:47,568 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 16:04:47,571 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.0840 - accuracy: 0.9765

Test accuracy: 97.6%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 16:04:48,117 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-07 16:04:48,119 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-07 16:04:48,121 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-07 16:04:48,124 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-07 16:04:48,127 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:04:48,129 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-07 16:04:48,132 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisione

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 5s 12ms/step - loss: 0.1083 - accuracy: 0.9683
Epoch 2/5
157/157 [==============================] - 5s 12ms/step - loss: 0.1130 - accuracy: 0.9654
Epoch 2/5
157/157 [==============================] - 2s 13ms/step - loss: 0.0983 - accuracy: 0.9699
Epoch 3/5
157/157 [==============================] - 2s 13ms/step - loss: 0.1000 - accuracy: 0.9689
Epoch 3/5
157/157 [==============================] - 2s 13ms/step - loss: 0.1004 - accuracy: 0.9689
Epoch 3/5
157/157 [==============================] - 2s 12ms/step - loss: 0.0921 - accuracy: 0.9707
Epoch 4/5
Epoch 4/5
157/157 [==============================] - 2s 12ms/step - loss: 0.0887 - accuracy: 0.9719
Epoch 4/5
157/157 [==============================] - 2s 11ms/step - loss: 0.0809 - accuracy: 0.9742
Epoch 5/5
157/157 [==============================] - 2s 11ms/step - loss: 0.0835 - accuracy: 0.9740
Epoch 5/5
Epoch 5/5
157/157 [==============================] - 2s 14ms

2023-07-07 16:05:04,363 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 16:05:04,368 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
2023-07-07 16:05:04,397 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 16:05:04,398 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 16:05:04,400 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-

sending data to divider
sending data to divider
sending data to divider


2023-07-07 16:05:04,590 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-07 16:05:04,592 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
DEBUG:DividerAmbassador:127.0.0.1:50151
2023-07-07 16:05:04,593 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
DEBUG:DividerAmbassador:127.0.0.1:50152
2023-07-07 16:05:04,594 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 1
2023-07-07 16:05:04,595 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 2
DEBUG:DividerAmbassador:127.0.0.1:50153
2023-07-07 16:05:04,597 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 2 to worker 3
DEBUG:DividerAmbassador:divider begins will not send data in iteration 2 to worker 1
2023-07-07 16:05:04,600 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/1.pkl to worker1
DEBUG:DividerAmbassador:divider begins will not send data in iteration 2 to worker 2
2023-07-07 16:05:04,604 [DEBUG] [DividerAmbassador] Sending ../

...

2023-07-07 16:05:27,313 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 16:05:27,316 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2


sending data to divider


2023-07-07 16:05:27,329 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 16:05:27,333 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-07 16:05:27,362 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 16:05:27,399 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 16:05:27,412 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 16:05:27,447 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-07 16:05:27,450 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 16:05:27,451 [DEBUG] 

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.0691 - accuracy: 0.9815

Test accuracy: 98.2%


2023-07-07 16:06:08,283 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 16:06:08,292 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
DEBUG:Coordinator:coordinator is sending the number of workers to provisioner
2023-07-07 16:06:08,577 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-07 16:06:08,577 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1
2023-07-07 16:06:08,851 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-07 16:06:08,851 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-07 16:06:08,856 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
2023-07-07 16:06:08,856 [INFO] [Worker_50153] Running the worker w

sending data to divider
sending data to divider


2023-07-07 16:06:10,338 [ERROR] [Worker_50153] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
2023-07-07 16:06:10,338 [ERROR] [Worker_50153] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
ERROR:Worker_50153:Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
2023-07-07 16:07:47,820 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
DEBUG:Coordinator:coordinator is sending the number of workers to provisioner
2023-07-07 16:07:47,913 [DEBUG] [Provisioner] Received '' from the coordinator to send status
DEBUG:Provisioner:Received '' from the coordinator to send status
2023-07-07 16:07:47,915 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
DEBUG:Provisioner:Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], st

sending data to divider
sending data to divider


2023-07-07 16:07:48,484 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
2023-07-07 16:07:48,484 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
2023-07-07 16:07:48,486 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-07 16:07:48,488 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-07 16:07:48,486 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-07 16:07:48,488 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
INFO:Worker_50153:Running the worker with id: 3 on iteration: 2
INFO:Worker_50152:Running the worker with id: 2 on iteration: 2
INFO:Worker_50151:Running the worker with id: 1 on iteration: 2


sending data to divider
sending data to divider


2023-07-07 16:07:49,423 [ERROR] [Worker_50152] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
2023-07-07 16:07:49,423 [ERROR] [Worker_50152] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
ERROR:Worker_50152:Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
2023-07-07 16:11:48,539 [DEBUG] [Coordinator] coordinator is sending workers info to divider
DEBUG:Coordinator:coordinator is sending workers info to divider
2023-07-07 16:11:48,544 [DEBUG] [Provisioner] Received '' from the coordinator to send status
DEBUG:Provisioner:Received '' from the coordinator to send status
2023-07-07 16:11:48,548 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
DEBUG:Provisioner:Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [

sending data to divider


2023-07-07 16:11:49,824 [ERROR] [Worker_50152] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
2023-07-07 16:11:49,824 [ERROR] [Worker_50152] Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
ERROR:Worker_50152:Error executing the file: shapes (785,785) and (9,) not aligned: 785 (dim 1) != 9 (dim 0)
